#Progetto NLP: Classificazione Email (Spam/Non-Spam), Estrazione di Entità (NER) e Riconoscimento di Topic

In [ ]:
!pip -q install gensim
!pip -q install spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import string
import spacy
import spacy.displacy as displacy
import nltk
from pprint import pprint
from scipy.stats import loguniform
from nltk.corpus import stopwords
from scipy import spatial
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report
from gensim.models import LdaMulticore
from gensim.corpora import Dictionary
from gensim.models import Word2Vec
from typing import List, Tuple, Any, Dict

###Parametri di configurazione e caricamento dei dati.

In [ ]:
URL_Dataset = "https://raw.githubusercontent.com/ProfAI/natural-language-processing/refs/heads/main/datasets/Verifica%20Finale%20-%20Spam%20Detection/spam_dataset.csv"
LABELS_NAME = ['non_spam', 'spam']
NUM_STEPS = 20 #iterazione del modello LDA
NUM_TOPICS = 6 #numero di topic da estrarre
W2V_VECTOR_SIZE = 200 #size del vettore w2v
TOP_N_WORDS = 10 #numero di parole chiave per ogni topic

CUSTOM_STOPWORDS = {
    'subject', 'meter', 'main', 'com', 'www', 'http', 'get', 'org', 'net' 'want', 'message'
}

CUSTOM_STOPWORDS_NER = {
    'Subject', 're', 'main', 'Main', 'from', 'to', 'cc', 'bcc', 'sent', 'date', 'version',
    'content', 'type', 'charset', 'boundary', 'return', 'path',
    'unsubscribe', 'disclaimer', 'copyright', 'reserve', 'right',
    'sincerely', 'regards', 'p.s.', 'ps', 'p', 's',
    'plain', 'text', 'html', 'original', 'message',
    'mailto', 'www', 'com', 'org', 'net'
}

In [ ]:
def load_dataset():
    """
    Carica il dataset dall'URL specificato
    """
    try:
        df = pd.read_csv(URL_Dataset)
        return df
    except Exception as e:
        print("ERRORe nell'URL", e)

In [ ]:
df = load_dataset()
df = df.drop(columns=['Unnamed: 0', 'label'])
print(f"Le email non spam sono: {df['label_num'].value_counts()[0]}| Le email di spam sono: {df['label_num'].value_counts()[1]}\n")
df.head()

Le email non spam sono: 3672| Le email di spam sono: 1499



,text,label_num
0,Subject: enron methanol ; meter # : 988291\nth...,0
1,"Subject: hpl nom for january 9 , 2001\n( see a...",0
2,"Subject: neon retreat\nho ho ho , we ' re arou...",0
3,"Subject: photoshop , windows , office . cheap ...",1
4,Subject: re : indian springs\nthis deal is to ...,0


Come potevamo aspettarci le mail non sono perfettamente bilanciate, e ovviamente abbiamo meno email di spam.

###Preprocessing del testo e Tokenizzazione.

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
english_stopwords = stopwords.words('english')
nlp = spacy.load('en_core_web_sm')
punctuation = set(string.punctuation)

In [ ]:
CONTRACTION_MAP = {
    "don't": "do not",
    "can't": "cannot",
    "won't": "will not",
    "it's": "it is",
    "i'm": "i am",
    "i'll": "i will",
    "i've": "i have",
    "i'd": "i would",
    "u're": "you are",
    "u've": "you have",
    "u'll": "you will",
    "u'd": "you would",
    "you're": "you are"
}

def expand_contractions(text: str, contraction_mapping: Dict[str, str]) -> str:
    """
    Sostituisce le contrazioni presenti nel testo con la loro forma estesa.

    Args:
        text (str): Testo contenente potenziali contrazioni.
        contraction_mapping (dict): Dizionario di mappatura contrazione -> espansione.

    Returns:
        str: Testo con le contrazioni espanse e gli apostrofi rimanenti rimossi.
    """

    for contraction, expansion in contraction_mapping.items():
        text = re.sub(re.escape(contraction), expansion, text)

    text = re.sub(r"'", '', text)
    return text

In [ ]:
def clean_text(text: str, ner: bool = False) -> str:
    """
    Applica una serie di operazioni di preprocessing al testo.

    Args:
        text (str): Testo da preprocessare.
        ner (bool): Flag che indica se il preprocessing è per NER (default: False).

    Returns:
        str: Testo preprocessato.
    """
    if(not ner):
        text = str(text).lower() #Trasforma il testo in minuscolo
        text = re.sub(r'http\S+', '', text) #Rimuove URL
        text = re.sub(r'&[a-z]+;', '', text) #Rimuove HTML
        text = re.sub('\n', ' ', text) #Rimuove i ritorno a capo
        for c in string.punctuation:
            text = text.replace(c, " ") #rimuove la punteggiatura
        document = nlp(text)
        sentence = ' '.join(token.lemma_ for token in document)
        sentence = ' '.join(word for word in sentence.split() if word not in english_stopwords and word not in CUSTOM_STOPWORDS and len(word) > 2)
        sentence = re.sub(r'\d+', '', sentence)
        return sentence.strip()
    else:
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'&[a-z]+;', '', text)
        text = re.sub('\n', ' ', text)
        text = re.sub(r'(\s|^)[%s]+|[ %s]+(\s|$)' % (re.escape(string.punctuation), re.escape(string.punctuation)), ' ', text)
        text = ' '.join(word for word in text.split() if word not in CUSTOM_STOPWORDS_NER and len(word) > 1)
        return text

In [ ]:
X, y = df['text'], df['label_num']

In [ ]:
X = X.apply(expand_contractions, contraction_mapping=CONTRACTION_MAP)
X = X.apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(f"\nDimensioni dei set: \nTrain: {len(X_train)}\nTest: {len(X_test)}")

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
vocabulary_size = len(tfidf_vectorizer.vocabulary_)
print(f"Dimensioni del vocabolario: {vocabulary_size}")


Dimensioni dei set: 
Train: 4136
Test: 1035
Dimensioni del vocabolario: 36460


###Costruzione del classificatore e valutazione

In [ ]:
def create_classifier(X_train: pd.DataFrame, y_train: pd.Series) -> LogisticRegression:
    """
    Crea un classificatore binario (SPAM/NON SPAM) utilizzando una regressione logistica.

    Args:
        X_train (pd.DataFrame): Set di dati di addestramento.
        y_train (pd.Series): Etichette corrispondenti ai dati di addestramento.

    Returns:
        LogisticRegression: Classificatore addestrato.
    """
    classifier = LogisticRegression(max_iter=1000)

    params = {
        "penalty": ['l1', 'l2'],
        "class_weight": ['balanced', None],
        "solver": ['liblinear', 'saga'],
        "C": loguniform(1e-4, 1e2)
    }

    rs = RandomizedSearchCV(
        estimator=classifier,
        param_distributions=params,
        n_iter=10,
        cv=5,
        random_state=0,
        n_jobs=-1
    )

    rs.fit(X_train, y_train)
    best_model = rs.best_estimator_
    return best_model


In [ ]:
def evaluate_classifier(classifier: LogisticRegression, X_true: pd.DataFrame, y_true: pd.Series) -> None:
    """
    Valuta un classificatore

    Args:
        classifier (LogisticRegression): Classificatore addestrato.
        X_true (pd.DataFrame): Set di dati di test.
        y_true (pd.Series): Etichette corrispondenti ai dati di test.

    """

    y_pred = classifier.predict(X_true)
    print(classification_report(y_true, y_pred, target_names=LABELS_NAME))

In [ ]:
classifier = create_classifier(X_train_tfidf, y_train)

In [ ]:
evaluate_classifier(classifier, X_train_tfidf, y_train)

              precision    recall  f1-score   support

    non_spam       1.00      1.00      1.00      2937
        spam       1.00      1.00      1.00      1199

    accuracy                           1.00      4136
   macro avg       1.00      1.00      1.00      4136
weighted avg       1.00      1.00      1.00      4136



In [ ]:
evaluate_classifier(classifier, X_test_tfidf, y_test)

              precision    recall  f1-score   support

    non_spam       0.99      0.98      0.99       735
        spam       0.96      0.98      0.97       300

    accuracy                           0.98      1035
   macro avg       0.98      0.98      0.98      1035
weighted avg       0.98      0.98      0.98      1035



In [ ]:
sentence_spam = "Subject: Re: [Newsgroups] Computer drugs vous vluesz und pill computers by microsoft"
classifier.predict(tfidf_vectorizer.transform([clean_text(sentence_spam)]))

array([1])

Dati per Topic Modeling e Ner

In [ ]:
data_topic = X.iloc[np.where(y == 1)].apply(lambda x: x.split())
data_ner = df['text'].iloc[np.where(y == 0)].apply(lambda x: clean_text(x, ner=True))

###Costruzione dei modelli LDA e Word2Vec per definire i Topic delle mail di spam

In [ ]:
def create_topic_model(data: pd.Series, num_topics: int = 5):
    """
    Crea un modello LDA per il topic modeling dei dati.

    Args:
        data (pd.Series): Serie di dati da processare.
        num_topics (int): Numero di topic da estrarre (default: 5).

    Returns:
        Tuple[LdaMulticore, Dictionary, List[List[int]]]: Tupla contenente il modello LDA, il dizionario e la lista di corpus.
    """

    print("Creazione del topic model...")

    #Creiamo il dizionario
    id2word = Dictionary(data)

    #creiamo il corpus
    corpus = [id2word.doc2bow(word) for word in data]

    lda_model = LdaMulticore(
        corpus=corpus,
        id2word=id2word,
        num_topics=num_topics,
        passes=NUM_STEPS,
        random_state=0
    )

    pprint(lda_model.print_topics())
    return lda_model, id2word, corpus

In [ ]:
def create_word2vec_model(data: pd.Series, vector_size: int = W2V_VECTOR_SIZE) -> Word2Vec:
    """
    Crea un modello Word2Vec per il topic modeling dei dati.

    Args:
        data (pd.Series): Serie di dati da processare.
        vector_size (int): Dimensione del vettore Word2Vec (default: 200).

    Returns:
        Word2Vec: Modello Word2Vec addestrato.
    """
    print("Creazione del modello Word2Vec...")

    w2v_model = Word2Vec(
        sentences=data,
        vector_size=vector_size,
        window=5,
        min_count=8,
        epochs=10,
        sg=1
    )

    #w2v_model.init_sims(replace=True)
    print(f"Word2Vec addestrato con {len(w2v_model.wv.index_to_key)} parole nel vocabolario.")
    return w2v_model


In [ ]:
lda_model = create_topic_model(data_topic, num_topics=NUM_TOPICS)
print("*"*100)
w2v_model = create_word2vec_model(data_topic)

Creazione del topic model...
[(0,
  '0.002*"und" + 0.002*"citibank" + 0.002*"artprice" + 0.001*"personal" + '
  '0.001*"pour" + 0.001*"vous" + 0.001*"ich" + 0.001*"email" + '
  '0.001*"research" + 0.001*"please"'),
 (1,
  '0.015*"nbsp" + 0.003*"microsoft" + 0.003*"pro" + 0.003*"download" + '
  '0.002*"rnd" + 0.002*"email" + 0.002*"review" + 0.002*"alt" + '
  '0.002*"instant" + 0.002*"sofftwaare"'),
 (2,
  '0.019*"font" + 0.013*"height" + 0.011*"width" + 0.009*"size" + '
  '0.008*"align" + 0.007*"color" + 0.007*"border" + 0.007*"face" + '
  '0.007*"href" + 0.006*"price"'),
 (3,
  '0.012*"company" + 0.007*"statement" + 0.006*"information" + 0.006*"stock" + '
  '0.005*"may" + 0.005*"price" + 0.005*"security" + 0.004*"investment" + '
  '0.004*"report" + 0.004*"please"'),
 (4,
  '0.005*"good" + 0.004*"email" + 0.004*"online" + 0.004*"click" + 0.004*"new" '
  '+ 0.004*"time" + 0.004*"well" + 0.004*"need" + 0.003*"take" + 0.003*"free"'),
 (5,
  '0.014*"pill" + 0.003*"viagra" + 0.002*"health" 

###Calcoliamo la distanza semantica tra i topics ottenuti per valutare l'eterogeneità dei contenuti delle email SPAM

In [ ]:
def get_topic_mean_vector(topic_id: int, lda_model: LdaMulticore, w2v_model: Word2Vec, topn: int = TOP_N_WORDS) -> np.ndarray:
    """
    Calcola il vettore medio del topic come media dei vettori Word2Vec
    delle sue top N parole chiave.

    Args:
        topic_id (int): ID del topic.
        lda_model (LdaMulticore): Modello LDA.
        w2v_model (Word2Vec): Modello Word2Vec.
        topn (int): Numero di parole chiave da considerare nel topic

    Returns:
        np.ndarray: Vettore medio del topic.
    """
    topic_words = [word for word, prob in lda_model.show_topic(topic_id, topn=topn)]
    vectors = []

    for word in topic_words:
        #Recupera il vettore solo se la parola è nel vocabolario Word2Vec
        if word in w2v_model.wv:
            vectors.append(w2v_model.wv[word])

    if vectors:
        #Calcolo del vettore medio
        return np.mean(vectors, axis=0)
    else:
        return None

In [ ]:
def similarity(vector1, vector2) -> float:
    """
    Calcola la similarità tra due vettori utilizzando la distanza coseno.

    Args:
        vector1 (np.ndarray): Primo vettore.
        vector2 (np.ndarray): Secondo vettore.

    Returns:
        float: Similarità tra i due vettori.
    """
    return round((1 - spatial.distance.cosine(vector1, vector2)),2)

In [ ]:
def show_similarity(id_topics, doc_mean_vectors) -> None:
    """
    Calcola e stampa la similarità tra i vettori dei topic.

    Args:
        id_topics (List[int]): Lista degli ID dei topic.
        doc_mean_vectors (List[np.ndarray]): Lista dei vettori medio dei topic.

    """
    print("Similitudine tra i topic:")
    no_repeat_topic = []
    for i in range(len(id_topics)):
        for j in range(len(id_topics)):
            if i != j and j not in no_repeat_topic:
                print(f"-Similitudine tra il topic {id_topics[i]} e il topic {id_topics[j]}: {similarity(doc_mean_vectors[i], doc_mean_vectors[j]):.3f}")
        no_repeat_topic.append(i)

In [ ]:
id_topics = [i for i in range(NUM_TOPICS)]
doc_mean_vectors = [get_topic_mean_vector(id, lda_model[0], w2v_model) for id in id_topics]

In [ ]:
show_similarity(id_topics, doc_mean_vectors)

Similitudine tra i topic:
-Similitudine tra il topic 0 e il topic 1: 0.680
-Similitudine tra il topic 0 e il topic 2: 0.550
-Similitudine tra il topic 0 e il topic 3: 0.520
-Similitudine tra il topic 0 e il topic 4: 0.590
-Similitudine tra il topic 0 e il topic 5: 0.530
-Similitudine tra il topic 1 e il topic 2: 0.670
-Similitudine tra il topic 1 e il topic 3: 0.500
-Similitudine tra il topic 1 e il topic 4: 0.690
-Similitudine tra il topic 1 e il topic 5: 0.720
-Similitudine tra il topic 2 e il topic 3: 0.380
-Similitudine tra il topic 2 e il topic 4: 0.560
-Similitudine tra il topic 2 e il topic 5: 0.550
-Similitudine tra il topic 3 e il topic 4: 0.590
-Similitudine tra il topic 3 e il topic 5: 0.550
-Similitudine tra il topic 4 e il topic 5: 0.660


###Analisi dei Topic e Metodologia di Similarità

L'obiettivo di questa fase è stato identificare automaticamente e separare le macro-categorie di contenuto all'interno del dataset. È stata adottata una metodologia in due passaggi: prima, il Topic Modeling (LDA) per l'identificazione dei concetti, e poi l'uso di Word Embeddings (Word2Vec) per la validazione semantica e il calcolo della similarità tra i topic.

Il modello LDA ha analizzato il corpus di testo pre-processato e ha identificato cinque (5) topic distinti. Ogni topic è rappresentato da una distribuzione di parole chiave, dove il peso (probabilità) indica la sua rilevanza all'interno del topic.

**Topic Identificati e Descrizione**:


Per esempio potremmo dire che:

- Il topic di indice 0 è MIXED

    Forte presenza di termini in lingua straniera (es. Tedesco: und, Francese: vous) e nomi di settori specifici come finanza e arte ecc.

- Il topic di indice 1 è TECH/SOFTWARE

    Dominato da termini relativi a tecnologia, software e prodotti digitali. La presenza di microsoft e download suggerisce offerte di prodotti, recensioni o problemi tecnici.

- Il topic di indice 2 è HTML/NOISE

    Caratterizzato quasi interamente da tag e attributi HTML (font, size, color, border, href, src). Questo topic non è semanticamente significativo, ma rappresenta il rumore strutturale dovuto alla formattazione delle email.

- Il topic di indice 3 è FINANCE

    Termini finanziari (stock, investment, security). Questo topic aggrega contenuti legati a mercati, offerte di investimento ecc.

- Il topic di indice 4 è GENERIC/SPAM

    Topic contenente parole generiche e comuni nelle comunicazioni di massa. La presenza di click, free e online indica un intento di "call-to-action" tipico dello spam commerciale non specifico.

- Il topic di indice 5 è PHARMA/HEALTH

    Contenuto Farmaceutico/Sanitario. Isola in modo pulito tutti i riferimenti a prodotti medici, pillole e problemi di salute, inclusi anche termini specifici di nicchia.


###Estraiamo dalle email NON SPAM le informazioni sulle Organizzazioni menzionate

In [ ]:
def extract_entities(sentence):
    """
    Estrae le entità nel testo utilizzando spaCy.

    Args:
        sentence (str): Testo da processare.

    Returns:
        dict: Dizionario contenente le entità estratte.
    """
    to_return = {"ORG": []}
    doc = nlp(sentence)

    for ent in doc.ents:
        if str(ent.label_) == 'ORG':
            to_return['ORG'].append(str(ent))

    #Togliamo i duplicati per un output più pulito
    to_return['ORG'] = sorted(list(set(to_return['ORG'])))

    return to_return

In [ ]:
for sentence in data_ner[:30]:
    print("-"*50)
    print(extract_entities(sentence))

--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': ['bearer']}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': []}
--------------------------------------------------
{'ORG': ['ect brian riley', 'ect nathan hlavaty hou', 'ect pat clynes corp enron', 'ees ees cynthia hakemack hou', 'tom acton corp enron']}
--------------------------------------------------
{'ORG': ['ect brian riley', 'mike morris corp enron enron']}
--------------------------------------------------
{'ORG': 